# Federated learning results

**MobileNetV1 1D alpha 1.00 · FedAvg · 25 local epochs per round · 10 global rounds · 3 seeds**

The fixed patient partition is **16 training, 2 development, and 2 test clients**, without patient overlap. The training pool contains **3,904,222 original prepared ECG beats** from 16 clients. **Three distinct training clients are selected randomly once and kept fixed across all rounds and all training seeds**, and every selected client uses all its original training beats. Each round selects **one random dev patient and one random test patient** from their existing two-patient pools. The cohort is client_01, client_09, and client_11 (selection seed 2024), containing 637,703 original training beats. Dev/test draws remain deterministic for each training seed/round. Both sample caps are `null`: no training/development subsampling applies. The production pipeline uses original ECG data, not artificial waveforms. Each chosen evaluation patient contributes all its data. Test identities are recorded in advance, but inference is deferred until all training/fine-tuning choices are locked. The round-specific dev patient supplies **S/V PR-AUC**, with the earliest round retained on ties. Test clients never participate in training or checkpoint selection.

This notebook uses the **.NET (C#)** kernel and reads saved JSON results only. **Run All does not train, build caches, load ECG signals, or evaluate test data.** Until results exist, tables display pending states. After a run has produced results, rerun the cells to refresh the report; a new snapshot is loaded each time the experiment cell runs. Unsaved outputs do not update automatically.

Edit `experimentDirectory` below to inspect a different experiment. The default is `artifacts/fl/mobilenetv1-fedavg-fixed3-10g-25l`. The report works before training and during partially completed experiments: only contiguous committed rounds are shown, with provisional checkpoints clearly labelled.

For a synthetic integration run, the experiment overview displays a **SYNTHETIC INTEGRATION CHECK ONLY** label. Such metrics verify execution and must not be interpreted as ECG model performance; the overview shows the actual short-run settings.

Because dev patients rotate, round-to-round PR-AUC differences also reflect patient/prevalence differences. The highest observed rotating-dev score is exploratory checkpoint selection, not a controlled comparison on a constant dev population.


In [ ]:
using System;
using System.IO;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
using System.Net;
using System.Text;
using System.Text.Json;

// Read-only report helpers. No training library, process execution, or signal reader.
public sealed class FlNotebookReport
{
    public readonly string DirectoryPath;
    readonly JsonElement config;
    readonly int[] seeds;
    readonly int rounds;
    readonly List<(int Seed, int Round, JsonElement Data)> history = new();
    readonly Dictionary<int, JsonElement> selected = new();
    readonly Dictionary<int, JsonElement> tests = new();
    static readonly string[] colors = { "#2563eb", "#d97706", "#059669" };
    static readonly string[] metrics = { "PR-AUC S/V", "ROC-AUC S/V", "Miss rate S/V", "FAR S/V", "Accuracy", "Macro-F1", "Loss", "Macro PR-AUC", "Macro ROC-AUC", "AP S/V" };

    public static string FindRepository()
    {
        for (var d = new DirectoryInfo(Environment.CurrentDirectory); d != null; d = d.Parent)
            if (File.Exists(Path.Combine(d.FullName, "implementation", "configs", "fedavg.json"))) return d.FullName;
        throw new DirectoryNotFoundException("Open this notebook from the repository, or set repositoryDirectory to its absolute path.");
    }

    static JsonElement Read(string path)
    {
        using var doc = JsonDocument.Parse(File.ReadAllText(path));
        return doc.RootElement.Clone();
    }
    static string H(object value) => WebUtility.HtmlEncode(Convert.ToString(value, CultureInfo.InvariantCulture) ?? "");
    static string F(double? n, bool percentage = false) => n.HasValue ? (percentage ? (n.Value * 100).ToString("F2", CultureInfo.InvariantCulture) + "%" : n.Value.ToString("F4", CultureInfo.InvariantCulture)) : "—";
    static double? Number(JsonElement e, string key) => e.TryGetProperty(key, out var v) && v.ValueKind == JsonValueKind.Number && v.TryGetDouble(out double n) && double.IsFinite(n) ? n : null;
    static double? Metric(JsonElement e, string label) => label switch {
        "PR-AUC S/V" => Number(e.GetProperty("RareSv"), "PrAuc"),
        "ROC-AUC S/V" => Number(e.GetProperty("RareSv"), "RocAuc"),
        "Miss rate S/V" => Number(e.GetProperty("RareSv"), "MissRate"),
        "FAR S/V" => Number(e.GetProperty("RareSv"), "Far"),
        "AP S/V" => Number(e.GetProperty("RareSv"), "AveragePrecision"),
        "Macro-F1" => Number(e, "MacroF1"), "Macro PR-AUC" => Number(e, "MacroPrAuc"),
        "Macro ROC-AUC" => Number(e, "MacroRocAuc"), _ => Number(e, label)
    };
    static bool Percent(string label) => label is "Miss rate S/V" or "FAR S/V" or "Accuracy";
    static string Row(IEnumerable<string> cells) => "<tr>" + string.Concat(cells.Select(c => "<td style='padding:6px;border-bottom:1px solid #e2e8f0'>" + H(c) + "</td>")) + "</tr>";
    static string Table(IEnumerable<string> headers, IEnumerable<string> rows) => "<div style='overflow-x:auto'><table style='border-collapse:collapse;white-space:nowrap'><thead><tr>"
        + string.Concat(headers.Select(h => "<th style='padding:7px;text-align:left;border-bottom:2px solid #94a3b8'>" + H(h) + "</th>"))
        + "</tr></thead><tbody>" + string.Concat(rows) + "</tbody></table></div>";
    static string Card(string message) => "<p style='padding:12px;border:1px solid #94a3b8;border-radius:6px'>" + H(message) + "</p>";
    static string Detail(string title, string body) => "<details><summary>" + H(title) + "</summary>" + body + "</details>";

    public FlNotebookReport(string experimentDirectory, string fallbackConfig)
    {
        DirectoryPath = Path.GetFullPath(experimentDirectory);
        config = Read(File.Exists(Path.Combine(DirectoryPath, "config.json")) ? Path.Combine(DirectoryPath, "config.json") : fallbackConfig);
        seeds = config.GetProperty("Seeds").EnumerateArray().Select(s => s.GetInt32()).ToArray();
        rounds = config.GetProperty("GlobalRounds").GetInt32();
        foreach (int seed in seeds)
        {
            string dir = Path.Combine(DirectoryPath, "seed-" + seed);
            // Only contiguous, atomically committed rounds enter any table or chart.
            for (int round = 1; round <= rounds; round++)
            {
                string path = Path.Combine(dir, $"round-{round:D3}");
                if (!File.Exists(Path.Combine(path, "global.bin")) || !File.Exists(Path.Combine(path, "metrics.json"))) break;
                var data = Read(Path.Combine(path, "metrics.json"));
                if (data.GetProperty("Seed").GetInt32() != seed || data.GetProperty("Round").GetInt32() != round) throw new InvalidDataException("Round identity mismatch: " + path);
                history.Add((seed, round, data));
            }
            if (File.Exists(Path.Combine(dir, "selected.json")))
            {
                var choice = Read(Path.Combine(dir, "selected.json"));
                if (history.Count(h => h.Seed == seed) != rounds || choice.GetProperty("Seed").GetInt32() != seed
                    || !history.Any(h => h.Seed == seed && h.Round == choice.GetProperty("BestRound").GetInt32()))
                    throw new InvalidDataException("Selected checkpoint has no complete seed history.");
                selected.Add(seed, choice);
            }
            if (File.Exists(Path.Combine(dir, "test.json")))
            {
                var test = Read(Path.Combine(dir, "test.json"));
                if (!selected.TryGetValue(seed, out var choice) || test.GetProperty("Seed").GetInt32() != seed
                    || test.GetProperty("BestRound").GetInt32() != choice.GetProperty("BestRound").GetInt32())
                    throw new InvalidDataException("Test result does not match the development-selected checkpoint.");
                tests.Add(seed, test);
            }
        }
    }

    public string Overview()
    {
        string statusPath = Path.Combine(DirectoryPath, "status.json");
        string status = File.Exists(statusPath) ? Read(statusPath).GetProperty("Status").GetString() ?? "unknown" : "not started";
        var values = new List<string> {
            Row(new[] { "Model", config.GetProperty("Model").GetString() ?? "unknown" }),
            Row(new[] { "Dataset scope", config.GetProperty("Name").GetString()?.StartsWith("integration-", StringComparison.Ordinal) == true ? "SYNTHETIC INTEGRATION CHECK ONLY — not patient performance" : "Prepared Icentia11k patient data" }),
            Row(new[] { "Schedule", $"{config.GetProperty("LocalEpochs").GetInt32()} local epochs per round × {rounds} global rounds × {seeds.Length} seeds" }),
            Row(new[] { "Partition", "16 training / 2 development / 2 test clients; fixed patient split" }),
            Row(new[] { "Training sample cap per client", config.TryGetProperty("TrainSamplesPerClient", out var trainCap) && trainCap.ValueKind == JsonValueKind.Number ? trainCap.ToString() + " (class-stratified)" : "Full source data" }),
            Row(new[] { "Development sample cap per client", config.TryGetProperty("DevSamplesPerClient", out var devCap) && devCap.ValueKind == JsonValueKind.Number ? devCap.ToString() + " (class-stratified)" : "Full source data" }),
            Row(new[] { "Final test scope", "All source examples; no test sampling or fitting" }),
            Row(new[] { "CPU convolution path", config.TryGetProperty("OptimizedCpuConvolutions", out var optimized) && optimized.ValueKind == JsonValueKind.True ? "Equivalent vectorized depthwise / matrix-multiplication pointwise" : "Native Conv1d" }),
            Row(new[] { "Seeds", string.Join(", ", seeds) }),
            Row(new[] { "Training clients per round", config.TryGetProperty("ClientsPerRound", out var participation) ? participation.ToString() + (config.TryGetProperty("ClientSelectionMode", out var mode) && mode.GetString() == "fixed" ? " fixed across rounds and seeds" : " randomly selected each round from 16") : "16 (legacy run)" }),
            Row(new[] { "Round evaluation", config.TryGetProperty("ClientsPerRound", out _) ? "One random dev and one preselected test patient; test inference deferred" : "Legacy evaluation protocol" }),
            Row(new[] { "FedAvg status", status }),
            Row(new[] { "Committed rounds", $"{history.Count}/{rounds * seeds.Length}" }),
            Row(new[] { "Selected seeds / test results", $"{selected.Count}/{seeds.Length} / {tests.Count}/{seeds.Length}" }),
            Row(new[] { "Experiment directory", DirectoryPath })
        };
        foreach (var stage in new[] { (File: "fine_tuning_status.json", Label: "Fine-tuning status"), (File: "test_status.json", Label: "Test evaluation status") })
        {
            string path = Path.Combine(DirectoryPath, stage.File);
            values.Add(Row(new[] { stage.Label, File.Exists(path) ? Read(path).GetProperty("Status").GetString() ?? "unknown" : "not started" }));
        }
        string cohortPath = Path.Combine(DirectoryPath, "cohort.json");
        if (File.Exists(cohortPath)) values.Add(Row(new[] { "Fixed training cohort", string.Join(", ", Read(cohortPath).GetProperty("TrainingClients").EnumerateArray().Select(x => x.GetProperty("Id").GetString())) }));
        string clientsPath = Path.Combine(DirectoryPath, "clients.json");
        if (File.Exists(clientsPath))
        {
            var clients = Read(clientsPath).EnumerateArray().ToArray();
            foreach (string split in new[] { "train", "dev" })
                {
                    values.Add(Row(new[] { split + " selected examples", clients.Where(c => c.GetProperty("Split").GetString() == split).Sum(c => c.GetProperty("Count").GetInt64()).ToString("N0", CultureInfo.InvariantCulture) }));
                    values.Add(Row(new[] { split + " source examples", clients.Where(c => c.GetProperty("Split").GetString() == split).Sum(c => c.TryGetProperty("SourceCount", out var count) ? count.GetInt64() : c.GetProperty("Count").GetInt64()).ToString("N0", CultureInfo.InvariantCulture) }));
                }
        }
        return Table(new[] { "Setting", "Value" }, values) + (history.Count == 0 ? Card("No training results yet. This notebook reads saved results only; it never starts training or test evaluation.") : "");
    }

    public string Summary()
    {
        var rows = new List<string>();
        foreach (int seed in seeds)
        {
            var available = history.Where(h => h.Seed == seed).ToArray();
            if (available.Length == 0) { rows.Add(Row(new[] { seed.ToString(), "Development", "Pending", "—" }.Concat(metrics.Select(_ => "—")))); continue; }
            bool final = selected.TryGetValue(seed, out var choice);
            var best = available.OrderByDescending(h => Metric(h.Data.GetProperty("Dev"), "PR-AUC S/V")).ThenBy(h => h.Round).First();
            int round = final ? choice.GetProperty("BestRound").GetInt32() : best.Round;
            var dev = final ? choice.GetProperty("Dev") : best.Data.GetProperty("Dev");
            rows.Add(Row(new[] { seed.ToString(), "Development", final ? "Selected" : $"Provisional ({available.Length}/{rounds} rounds)", round.ToString() }
                .Concat(metrics.Select(m => F(Metric(dev, m), Percent(m))))));
            if (tests.TryGetValue(seed, out var test))
                rows.Add(Row(new[] { seed.ToString(), "Test", "Saved evaluation", round.ToString() }.Concat(metrics.Select(m => F(Metric(test.GetProperty("Pooled"), m), Percent(m))))));
        }
        string table = Table(new[] { "Seed", "Split", "Checkpoint", "Round" }.Concat(metrics), rows);
        var averages = new List<string>();
        foreach (string split in new[] { "Development", "Test" })
        {
            var evaluations = split == "Development" ? selected.Values.Select(s => s.GetProperty("Dev")).ToArray() : tests.Values.Select(t => t.GetProperty("Pooled")).ToArray();
            foreach (string metric in metrics)
            {
                var numbers = evaluations.Select(e => Metric(e, metric)).Where(v => v.HasValue).Select(v => v.GetValueOrDefault()).ToArray();
                double? mean = numbers.Length == 0 ? null : numbers.Average();
                double? sd = mean.HasValue ? Math.Sqrt(numbers.Select(v => Math.Pow(v - mean.Value, 2)).Average()) : null;
                averages.Add(Row(new[] { split, metric, F(mean, Percent(metric)), F(sd, Percent(metric)), $"{numbers.Length}/{seeds.Length}" }));
            }
        }
        return table + "<p>Aggregates below include only published development-selected checkpoints and saved test evaluations. Provisional seeds are excluded. SD is the population standard deviation across available seeds; a single seed is not evidence of stability.</p>"
            + Detail("Across-seed metrics: mean and standard deviation", Table(new[] { "Split", "Metric", "Mean", "SD", "Defined seeds" }, averages));
    }

    public string Charts()
    {
        if (history.Count == 0) return Card("Development curves will appear after the first complete global round.");
        var svg = new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 960 570' role='img' aria-label='Development S/V metrics by global round' style='width:100%;max-width:960px;background:white;color:#172033;font-family:sans-serif'><title>Development metrics by global round</title>");
        string N(double x) => x.ToString("0.###", CultureInfo.InvariantCulture);
        for (int panel = 0; panel < 4; panel++)
        {
            double left = 65 + (panel % 2) * 480, top = 42 + (panel / 2) * 255, width = 385, height = 165;
            double X(int r) => left + (rounds == 1 ? width / 2 : (r - 1.0) * width / (rounds - 1));
            double Y(double v) => top + height * (1 - v);
            svg.Append($"<text x='{N(left)}' y='{N(top - 15)}' font-size='16' font-weight='bold'>{H(metrics[panel])}</text>");
            foreach (double tick in new[] { 0.0, .5, 1.0 })
                svg.Append($"<line x1='{N(left)}' x2='{N(left + width)}' y1='{N(Y(tick))}' y2='{N(Y(tick))}' stroke='#e2e8f0'/><text x='{N(left - 12)}' y='{N(Y(tick) + 4)}' text-anchor='end' font-size='11'>{N(tick)}</text>");
            foreach (int tick in new[] { 1, Math.Max(1, (rounds + 1) / 2), rounds }.Distinct())
                svg.Append($"<text x='{N(X(tick))}' y='{N(top + height + 20)}' text-anchor='middle' font-size='11'>{tick}</text>");
            svg.Append($"<text x='{N(left + width / 2)}' y='{N(top + height + 40)}' text-anchor='middle' font-size='12'>Global round</text>");
            for (int i = 0; i < seeds.Length; i++)
            {
                string color = colors[i % colors.Length];
                var points = history.Where(h => h.Seed == seeds[i]).OrderBy(h => h.Round).Select(h => (h.Round, Value: Metric(h.Data.GetProperty("Dev"), metrics[panel]))).Where(p => p.Value.HasValue).ToArray();
                if (points.Length == 0) continue;
                svg.Append($"<polyline fill='none' stroke='{color}' stroke-width='2' points='{string.Join(" ", points.Select(p => N(X(p.Round)) + "," + N(Y(p.Value.GetValueOrDefault()))))}'/>");
                foreach (var p in points) svg.Append($"<circle cx='{N(X(p.Round))}' cy='{N(Y(p.Value.GetValueOrDefault()))}' r='2.6' fill='{color}'><title>Seed {seeds[i]}, round {p.Round}: {F(p.Value)}</title></circle>");
            }
        }
        for (int i = 0; i < seeds.Length; i++) svg.Append($"<rect x='{65 + i * 180}' y='545' width='15' height='4' fill='{colors[i % colors.Length]}'/><text x='{88 + i * 180}' y='551' font-size='12'>Seed {seeds[i]}</text>");
        return svg.Append("</svg>").ToString();
    }

    public string Details()
    {
        if (history.Count == 0) return Card("Per-class metrics, patient results, and confusion matrices are pending.");
        var result = new StringBuilder();
        void Append(int seed, int round, string split, JsonElement evaluation, JsonElement byClient, bool final)
        {
            var classes = evaluation.GetProperty("Classes").EnumerateArray().Select(c => Row(new[] {
                c.GetProperty("Label").GetString() ?? "?", c.GetProperty("Support").ToString(),
                F(Number(c, "Precision")), F(Number(c, "Recall")), F(Number(c, "F1")),
                F(Number(c, "PrAuc")), F(Number(c, "AveragePrecision")), F(Number(c, "RocAuc")),
                F(Number(c, "MissRate"), true), F(Number(c, "Far"), true) }));
            string body = Table(new[] { "Class", "Support", "Precision", "Recall", "F1", "PR-AUC", "AP", "ROC-AUC", "Miss rate", "FAR" }, classes);
            body += Table(new[] { "Client", "Examples" }.Concat(metrics), byClient.EnumerateObject().OrderBy(p => p.Name).Select(p =>
                Row(new[] { p.Name, p.Value.GetProperty("Count").ToString() }.Concat(metrics.Select(m => F(Metric(p.Value, m), Percent(m)))))));
            string[] labels = { "N", "S", "V", "Q" }; int row = 0;
            body += "<p>Confusion matrix: rows are true classes; columns are predicted classes.</p>" + Table(new[] { "True / predicted", "N", "S", "V", "Q" },
                evaluation.GetProperty("Confusion").EnumerateArray().Select(r => Row(new[] { labels[row++] }.Concat(r.EnumerateArray().Select(v => v.ToString())))));
            var rare = evaluation.GetProperty("RareSv");
            body += Table(new[] { "S/V binary metric", "Value" }, new[] { "PositiveSupport", "NegativeSupport", "Threshold", "Tp", "Fp", "Tn", "Fn" }.Select(k => Row(new[] { k, rare.GetProperty(k).ToString() })));
            body += "<p>Defined one-vs-rest AUC classes: PR " + H(evaluation.GetProperty("DefinedPrClasses")) + "/4; ROC " + H(evaluation.GetProperty("DefinedRocClasses")) + "/4. Undefined class metrics are excluded from macro AUC.</p>";
            result.Append(Detail($"Seed {seed} · {split} · round {round} · {(final ? "selected checkpoint" : "provisional best completed round")}", body));
        }
        foreach (int seed in seeds)
        {
            var available = history.Where(h => h.Seed == seed).ToArray(); if (available.Length == 0) continue;
            bool final = selected.TryGetValue(seed, out var choice);
            int round = final ? choice.GetProperty("BestRound").GetInt32() : available.OrderByDescending(h => Metric(h.Data.GetProperty("Dev"), "PR-AUC S/V")).ThenBy(h => h.Round).First().Round;
            var best = available.Single(h => h.Round == round);
            Append(seed, round, "Development", best.Data.GetProperty("Dev"), best.Data.GetProperty("DevByClient"), final);
            if (tests.TryGetValue(seed, out var test)) Append(seed, round, "Test", test.GetProperty("Pooled"), test.GetProperty("ByClient"), true);
        }
        return result.ToString();
    }

    public string AllRounds()
    {
        if (history.Count == 0) return Card("Global-round and local-client tables will be populated from completed rounds.");
        var rows = history.Select(h => Row(new[] { h.Seed.ToString(), h.Round.ToString() }.Concat(metrics.Select(m => F(Metric(h.Data.GetProperty("Dev"), m), Percent(m))))
            .Concat(new[] { F(Number(h.Data, "Seconds")) })));
        string result = Detail("All completed global rounds", Table(new[] { "Seed", "Round" }.Concat(metrics).Concat(new[] { "Seconds" }), rows));
        var locals = history.SelectMany(h => h.Data.GetProperty("Clients").EnumerateArray().Select(c => Row(new[] {
            h.Seed.ToString(), h.Round.ToString(), c.GetProperty("Client").GetString() ?? "?", c.GetProperty("Examples").ToString(),
            c.GetProperty("Epochs").ToString(), F(Number(c, "FinalLoss")), F(Number(c, "Seconds")) })));
        return result + Detail("Local client training: final-epoch loss and timing", Table(new[] { "Seed", "Round", "Client", "Distinct examples", "Local epochs", "Final loss", "Seconds" }, locals));
    }
    JsonElement[] SavedModelReports() => System.IO.Directory.Exists(Path.Combine(DirectoryPath, "model-reports"))
        ? System.IO.Directory.GetFiles(Path.Combine(DirectoryPath, "model-reports"), "*.json").OrderBy(p => p, StringComparer.Ordinal).Select(Read).ToArray()
        : Array.Empty<JsonElement>();

    public string ModelComparison()
    {
        var reports = SavedModelReports();
        if (reports.Length == 0) return Card("Global, local, and fine-tuned model comparisons will appear after the relevant stages finish.");
        string Name(JsonElement r, string field) => r.GetProperty(field).ValueKind == JsonValueKind.Null ? "server" : r.GetProperty(field).ToString();
        string Render(IEnumerable<JsonElement> entries) => Table(new[] { "Seed", "Model", "Training client", "Split", "Round", "Evaluation patient/client" }.Concat(metrics), entries.Select(r =>
            Row(new[] { Name(r, "Seed"), Name(r, "Kind"), Name(r, "Client"), Name(r, "Split"), Name(r, "Round"), string.Join(", ", r.GetProperty("ByClient").EnumerateObject().Select(p => p.Name)) }
                .Concat(metrics.Select(m => F(Metric(r.GetProperty("Pooled"), m), Percent(m)))))));
        string result = Render(reports.Where(r => Name(r, "Kind").StartsWith("global-", StringComparison.Ordinal)));
        result += Detail("All 16 local models before and after fine-tuning, by seed and split", Render(reports.Where(r => !Name(r, "Kind").StartsWith("global-", StringComparison.Ordinal))));
        var aggregates = new List<string>();
        foreach (var group in reports.GroupBy(r => (Kind: Name(r, "Kind"), Client: Name(r, "Client"), Split: Name(r, "Split"))))
        {
            var row = new List<string> { group.Key.Kind, group.Key.Client, group.Key.Split };
            foreach (string metric in metrics)
            {
                var values = group.Select(r => Metric(r.GetProperty("Pooled"), metric)).Where(v => v.HasValue).Select(v => v.GetValueOrDefault()).ToArray();
                double? mean = values.Length == 0 ? null : values.Average();
                double? sd = mean.HasValue ? Math.Sqrt(values.Select(v => Math.Pow(v - mean.Value, 2)).Average()) : null;
                row.Add(F(mean, Percent(metric)) + " ± " + F(sd, Percent(metric)) + $" ({values.Length}/{seeds.Length})");
            }
            aggregates.Add(Row(row));
        }
        result += Detail("Mean ± population SD across seeds, separately for each model/client/split", Table(new[] { "Model", "Training client", "Split" }.Concat(metrics), aggregates));
        foreach (var r in reports)
        {
            var e = r.GetProperty("Pooled");
            string body = Table(new[] { "Evaluation patient/client", "Examples" }.Concat(metrics), r.GetProperty("ByClient").EnumerateObject().Select(p =>
                Row(new[] { p.Name, p.Value.GetProperty("Count").ToString() }.Concat(metrics.Select(m => F(Metric(p.Value, m), Percent(m)))))));
            string[] fields = { "Precision", "Recall", "F1", "PrAuc", "AveragePrecision", "RocAuc", "MissRate", "Far" };
            body += Table(new[] { "Class", "Support" }.Concat(fields), e.GetProperty("Classes").EnumerateArray().Select(c =>
                Row(new[] { c.GetProperty("Label").ToString(), c.GetProperty("Support").ToString() }.Concat(fields.Select(f => F(Number(c, f), f is "MissRate" or "Far"))))));
            string[] labels = { "N", "S", "V", "Q" }; int row = 0;
            body += Table(new[] { "True / predicted", "N", "S", "V", "Q" }, e.GetProperty("Confusion").EnumerateArray().Select(c =>
                Row(new[] { labels[row++] }.Concat(c.EnumerateArray().Select(v => v.ToString())))));
            result += Detail($"Details: seed {Name(r, "Seed")} · {Name(r, "Kind")} · {Name(r, "Client")} · {Name(r, "Split")}", body);
        }
        return result;
    }

    public string FineTuningResults()
    {
        var results = new List<JsonElement>();
        foreach (int seed in seeds) for (int client = 0; client < 16; client++)
        {
            string path = Path.Combine(DirectoryPath, $"seed-{seed}", "fine-tuning", $"client_{client:D2}", "result.json");
            if (File.Exists(path)) results.Add(Read(path));
        }
        if (results.Count == 0) return Card("Local fine-tuning has not completed any searches. Candidate and selected-configuration tables are pending.");
        string summary = Table(new[] { "Seed", "Training client", "Selected configuration", "Layers", "Learning rate", "Epoch", "Source round", "Dev client", "Test client", "Dev PR-AUC before", "Dev PR-AUC after", "Dev PR-AUC change" }, results.Select(r => {
            double? before = Metric(r.GetProperty("BaselineDev"), "PR-AUC S/V"), after = Metric(r.GetProperty("Dev"), "PR-AUC S/V");
            return Row(new[] { r.GetProperty("Seed").ToString(), r.GetProperty("Client").ToString(), r.GetProperty("BestConfiguration").ToString(),
                r.GetProperty("Layers").ToString(), r.GetProperty("LearningRate").ToString(), r.GetProperty("Epochs").ToString(), r.TryGetProperty("SourceRound", out var sr) ? sr.ToString() : "legacy",
                r.TryGetProperty("DevClient", out var dc) ? dc.ToString() : "legacy", r.TryGetProperty("TestClient", out var tc) ? tc.ToString() : "legacy", F(before), F(after), F(after - before) });
        }));
        int eligible = history.GroupBy(h => h.Seed).Sum(g => g.SelectMany(h => h.Data.GetProperty("Clients").EnumerateArray()).Select(c => c.GetProperty("Client").GetString()).Distinct().Count());
        summary += $"<p>Completed searches: {results.Count}/{eligible} participating client/seed pairs from committed rounds. 'Unchanged' means no candidate improved the original local model. Each local selection uses its last participation round's assigned dev patient, never test scores.</p>";
        foreach (var r in results)
        {
            var candidates = r.GetProperty("Candidates").EnumerateArray().ToArray();
            string body = Table(new[] { "Candidate", "Layers", "Learning rate", "Best epoch" }.Concat(metrics), candidates.Select(c =>
                Row(new[] { c.GetProperty("Id").ToString(), c.GetProperty("Layers").ToString(), c.GetProperty("LearningRate").ToString(), c.GetProperty("BestEpoch").ToString() }
                    .Concat(metrics.Select(m => F(Metric(c.GetProperty("Dev"), m), Percent(m)))))));
            foreach (var c in candidates.Where(c => c.GetProperty("History").GetArrayLength() > 0))
                body += Detail(c.GetProperty("Id").ToString() + " — epoch history", Table(new[] { "Epoch", "Training loss" }.Concat(metrics),
                    c.GetProperty("History").EnumerateArray().Select(e => Row(new[] { e.GetProperty("Epoch").ToString(), F(Number(e, "TrainLoss")) }
                        .Concat(metrics.Select(m => F(Metric(e.GetProperty("Dev"), m), Percent(m))))))));
            summary += Detail($"Search: seed {r.GetProperty("Seed")} · {r.GetProperty("Client")}", body);
        }
        var reports = SavedModelReports();
        var paired = new List<string>();
        foreach (var r in results)
        {
            int seed = r.GetProperty("Seed").GetInt32(); string client = r.GetProperty("Client").GetString() ?? "";
            foreach (string split in new[] { "dev", "test" })
            {
                var pair = reports.Where(p => p.GetProperty("Seed").GetInt32() == seed && p.GetProperty("Client").GetString() == client && p.GetProperty("Split").GetString() == split).ToArray();
                var before = pair.Where(p => p.GetProperty("Kind").GetString() == "local-final").ToArray();
                var after = pair.Where(p => p.GetProperty("Kind").GetString() == "local-fine-tuned").ToArray();
                if (before.Length != 1 || after.Length != 1) continue;
                foreach (string metric in metrics)
                {
                    double? b = Metric(before[0].GetProperty("Pooled"), metric), a = Metric(after[0].GetProperty("Pooled"), metric);
                    paired.Add(Row(new[] { seed.ToString(), client, split, metric, F(b, Percent(metric)), F(a, Percent(metric)), F(a - b, Percent(metric)) }));
                }
            }
        }
        return summary + Detail("Paired before/after comparison for every metric (test appears only after saved evaluation)",
            Table(new[] { "Seed", "Training client", "Split", "Metric", "Before", "After", "After − before" }, paired));
    }

    public string FineTuningChart()
    {
        var reports = SavedModelReports();
        if (!reports.Any(r => r.GetProperty("Kind").GetString() == "local-fine-tuned")) return Card("The local-model PR-AUC comparison will appear after fine-tuning.");
        var svg = new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 1100 730' role='img' aria-label='Local models before and after fine-tuning' style='width:100%;background:white;font-family:sans-serif'><title>Local-model S/V PR-AUC before and after fine-tuning</title>");
        string N(double v) => v.ToString("0.###", CultureInfo.InvariantCulture);
        for (int panel = 0; panel < 2; panel++)
        {
            string split = panel == 0 ? "dev" : "test";
            double left = 85 + panel * 545, top = 65, width = 390;
            svg.Append($"<text x='{left}' y='28' font-size='18'>{(panel == 0 ? "Development" : "Held-out test")} S/V PR-AUC</text>");
            foreach (double tick in new[] { 0.0, .25, .5, .75, 1.0 })
            {
                double x = left + width * tick;
                svg.Append($"<line x1='{N(x)}' x2='{N(x)}' y1='48' y2='673' stroke='#e2e8f0'/><text x='{N(x)}' y='692' text-anchor='middle' font-size='11'>{N(tick)}</text>");
            }
            for (int client = 0; client < 16; client++)
            {
                string id = $"client_{client:D2}"; double y = top + client * 38;
                svg.Append($"<text x='{N(left - 8)}' y='{N(y + 9)}' text-anchor='end' font-size='11'>{id}</text>");
                var pairedSeeds = seeds.Where(seed => reports.Any(r => r.GetProperty("Seed").GetInt32() == seed && r.GetProperty("Client").GetString() == id && r.GetProperty("Split").GetString() == split && r.GetProperty("Kind").GetString() == "local-final")
                    && reports.Any(r => r.GetProperty("Seed").GetInt32() == seed && r.GetProperty("Client").GetString() == id && r.GetProperty("Split").GetString() == split && r.GetProperty("Kind").GetString() == "local-fine-tuned")).ToArray();
                for (int kind = 0; kind < 2; kind++)
                {
                    string name = kind == 0 ? "local-final" : "local-fine-tuned", color = kind == 0 ? "#64748b" : "#0f766e";
                    var values = reports.Where(r => pairedSeeds.Contains(r.GetProperty("Seed").GetInt32()) && r.GetProperty("Client").GetString() == id && r.GetProperty("Split").GetString() == split && r.GetProperty("Kind").GetString() == name)
                        .Select(r => Metric(r.GetProperty("Pooled"), "PR-AUC S/V")).Where(v => v.HasValue).Select(v => v.GetValueOrDefault()).ToArray();
                    if (values.Length == 0) continue;
                    double mean = values.Average(), sd = Math.Sqrt(values.Select(v => Math.Pow(v - mean, 2)).Average()), yy = y + kind * 12;
                    svg.Append($"<rect x='{N(left)}' y='{N(yy)}' width='{N(width * mean)}' height='9' fill='{color}'><title>{id}, {name}, mean {F(mean)}, SD {F(sd)}, seeds {values.Length}</title></rect>");
                    double lo = left + width * Math.Max(0, mean - sd), hi = left + width * Math.Min(1, mean + sd);
                    svg.Append($"<line x1='{N(lo)}' x2='{N(hi)}' y1='{N(yy + 4.5)}' y2='{N(yy + 4.5)}' stroke='#172033'/>");
                }
                if (pairedSeeds.Length == 0) svg.Append($"<text x='{N(left + 10)}' y='{N(y + 9)}' font-size='11' fill='#64748b'>{(history.Any(h => h.Data.GetProperty("Clients").EnumerateArray().Any(c => c.GetProperty("Client").GetString() == id)) ? "pending" : "not selected")}</text>");
            }
        }
        return svg.Append("<rect x='85' y='714' width='14' height='8' fill='#64748b'/><text x='108' y='722' font-size='12'>Before fine-tuning</text><rect x='300' y='714' width='14' height='8' fill='#0f766e'/><text x='323' y='722' font-size='12'>After fine-tuning · mean ± SD across paired seeds</text></svg>").ToString();
    }
    public string RoundAssignments()
    {
        var rows = new List<string>(); var coverage = new List<string>(); var tests = new List<string>();
        foreach (int seed in seeds)
        {
            string seedDir = Path.Combine(DirectoryPath,$"seed-{seed}");
            string path = Path.Combine(seedDir,"schedule.json");
            if (!File.Exists(path)) continue;
            var selections = Read(path).EnumerateArray().ToArray();
            foreach (var selection in selections)
            {
                int round = selection.GetProperty("Round").GetInt32();
                rows.Add(Row(new[] { seed.ToString(),round.ToString(),string.Join(", ",selection.GetProperty("TrainClients").EnumerateArray().Select(x=>x.GetString())),
                    selection.GetProperty("DevClient").GetString() ?? "?",selection.GetProperty("TestClient").GetString() ?? "?",
                    history.Any(h=>h.Seed==seed && h.Round==round) ? "committed" : "planned" }));
            }
            string participation = Path.Combine(seedDir,"participation.json");
            if (File.Exists(participation)) foreach (var client in Read(participation).EnumerateArray())
                coverage.Add(Row(new[] { seed.ToString(),client.GetProperty("Client").GetString() ?? "?",client.GetProperty("Participations").ToString(),client.GetProperty("LastRound").ToString(),
                    client.GetProperty("Participations").GetInt32()==0 ? "Not selected; no local training or fine-tuning" : "Latest participation checkpoint retained" }));
            string testDir = Path.Combine(seedDir,"round-tests");
            if (System.IO.Directory.Exists(testDir)) foreach (string file in System.IO.Directory.GetFiles(testDir,"*.json").OrderBy(x=>x))
            {
                var r = Read(file);
                tests.Add(Row(new[] { seed.ToString(),r.GetProperty("Round").ToString(),r.GetProperty("Kind").ToString(),r.GetProperty("Client").GetString() ?? "server",
                    string.Join(", ",r.GetProperty("ByClient").EnumerateObject().Select(p=>p.Name)) }
                    .Concat(metrics.Select(m=>F(Metric(r.GetProperty("Pooled"),m),Percent(m))))));
            }
        }
        if (rows.Count==0) return Card("Round assignments will appear in schedule.json when an experiment is initialized.");
        return Table(new[] { "Seed","Round","Training clients","Dev patient/client","Preselected test patient/client","Round status" },rows)
            + Detail("Training-client participation and unselected clients",Table(new[] { "Seed","Client","Participations","Last round","Local status" },coverage))
            + Detail("Per-round test results — inference deferred until all tuning selections are locked",Table(new[] { "Seed","Round","Model","Training client","Evaluation patient/client" }.Concat(metrics),tests));
    }
}


In [ ]:
// This is the only path to change when inspecting another experiment.
var repositoryDirectory = FlNotebookReport.FindRepository();
var experimentDirectory = System.IO.Path.Combine(repositoryDirectory, "artifacts", "fl", "mobilenetv1-fedavg-fixed3-10g-25l");
var report = new FlNotebookReport(experimentDirectory,
    System.IO.Path.Combine(repositoryDirectory, "implementation", "configs", "fedavg.json"));
display(report.Overview(), "text/html");


## Results by seed

The table uses the same metrics as the model tests: **S/V PR-AUC, ROC-AUC, miss rate, FAR, and accuracy**. It also includes loss, macro-F1, macro PR-AUC/ROC-AUC, and S/V Average Precision. Expand the aggregate table for **mean ± population SD** and the number of defined seeds. Only published selected checkpoints contribute to the aggregate; incomplete seeds remain provisional.

S/V uses `p(S) + p(V)` with N/Q as negatives; Q retains its unclassifiable meaning. Miss rate = FN/(TP+FN); FAR = FP/(FP+TN). Their fixed threshold comes from the experiment configuration (default **0.5**). Accuracy and miss/FAR are displayed as percentages. PR-AUC uses trapezoidal integration; Average Precision (AP) is reported separately. Undefined metrics appear as **—**, without replacement by zero. The default full-data protocol retains original development and test prevalence. Compare before/after models within each split, not development AUC directly against test AUC or differently sampled smoke tests.


In [ ]:
display(report.Summary(), "text/html");


## Development metrics over global rounds

Each line represents one seed. PR-AUC and ROC-AUC are higher-is-better; miss rate and FAR are lower-is-better. Curves use the single dev patient selected for that round after each complete FedAvg round, with a common 0–1 scale. Test scores are never used in these curves or for checkpoint selection.


In [ ]:
display(report.Charts(), "text/html");


## Per-class and per-client results

Expand each seed/split for N/S/V/Q metrics, support, patient-level results, confusion matrices, and S/V binary counts. Per-class miss/FAR use one-vs-rest **argmax decisions**; grouped S/V miss/FAR use the probability threshold. Global metrics pool examples rather than average patient AUCs. Test sections appear only when the separate evaluation command has saved a matching `test.json`.


In [ ]:
display(report.Details(), "text/html");


## Complete round history and local training

Expandable tables retain all recorded global-round metrics and every local client's final-epoch loss, distinct example count, local epochs, and timing. Local loss uses training-only class weights and cannot be directly compared with unweighted development loss. The experiment records final local-epoch loss, not a full local-epoch history. Timing includes work within each round, including cache construction when needed.


In [ ]:
display(report.AllRounds(), "text/html");


## Global and local model results

`global-final` is the server model after round 10. `global-selected` is the checkpoint selected by development S/V PR-AUC. `local-final` is each participating client's last available local update, which can come from an earlier round. Clients never selected have no fitted local model and are excluded from fine-tuning. `local-fine-tuned` is that client's configuration selected after the fine-tuning search. Only the three participating local checkpoints are retained in each round.

**Evaluation protocol:** each local model trains only on its own training patient; its dev/test results use the single dev/test patients assigned to its last participation round. Each selected patient contributes its full original data. There are **no new within-client splits**. These scores measure transfer to unseen patients, not personalization on held-out data from that client's own patient. The 16 models share evaluation patients and are not 16 independent evaluation datasets.

Expand the tables for every local model's PR-AUC, ROC-AUC, miss rate, FAR, accuracy, loss, macro-F1, macro AUCs, AP, class metrics, patient metrics, and confusion matrix. Across-seed statistics are calculated separately for each model/client/split; clients are never pooled as independent seeds.


In [ ]:
display(report.ModelComparison(), "text/html");


## Local fine-tuning and configuration selection

After FedAvg, each last participating local model restarts from the **same saved local checkpoint** for every candidate. The default bounded grid tests **learning rates 0.001, 0.0001, 0.00001 × head-only or all layers**, for **up to 20 epochs per candidate**. The best epoch is part of the selected configuration. Head-only tuning freezes the feature extractor and its BatchNorm statistics. The saved experiment configuration defines the actual grid.

Every candidate trains on all original training examples from that local client only. The objective is the **local model's pooled S/V PR-AUC on the the single dev patient assigned to its last participation round**. The unchanged model is a candidate and wins a tie; other ties retain the first configuration and earliest epoch. No improvement is promised. Search is finite and completes the configured grid; it does not continue indefinitely until a desired score appears.

The selected weights and configuration are locked before test access. The **test patients receive inference only**, with no gradient updates, retraining, epoch selection, threshold selection, or configuration ranking. The search reuses the development set, so held-out test performance is the final check. Test tables remain pending until the explicit evaluation stage completes.


In [ ]:
display(report.FineTuningResults(), "text/html");


## Local PR-AUC before and after fine-tuning

Bars show mean S/V PR-AUC with population SD across the **same available seed pairs**, separately for each training client's model. Development and test are displayed separately. Missing results are marked pending. Development gains drive selection; test gains are measured without changing the chosen model.


In [ ]:
display(report.FineTuningChart(), "text/html");


## Fixed training cohort and deferred round tests

The three training clients are drawn once and remain identical for every round and training seed. The other 13 clients do not train or fine-tune. Dev/test patient assignments may vary per seed/round; test identities are recorded without reading signals, and all test inference is deferred until training/tuning are locked. `cohort.json` and the schedule tables make the selected identities explicit.


In [ ]:
display(report.RoundAssignments(), "text/html");
